# GroundingDINO Spatial Verification Test

Quick test notebook to verify:
1. GroundingDINO detects common objects (cat, vehicle, person, table, etc.)
2. Bounding box coordinates are correct (cx,cy,w,h → x1,y1,x2,y2)
3. Spatial geometry rules fire correctly (on/under, inside/outside, left/right)

**GPU:** T4 is fine. No API calls needed.

In [ ]:
!pip install -q transformers torch torchvision pillow accelerate

import torch, math
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import matplotlib.pyplot as plt
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Load GroundingDINO ────────────────────────────────────
gdino_id = "IDEA-Research/grounding-dino-tiny"
print(f"Loading {gdino_id}...")
gdino_proc  = AutoProcessor.from_pretrained(gdino_id)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_id).to(DEVICE)
print("✓ GroundingDINO loaded")

GDINO_TEXT_THRESHOLD = 0.20
GDINO_BOX_THRESHOLD  = 0.25


In [ ]:
# ── Detection ─────────────────────────────────────────────
def run_gdino(img: Image.Image, queries: list) -> dict:
    """Run GroundingDINO. Returns {query: [{box:[x1,y1,x2,y2], score, label}]}"""
    results = {}
    h, w = img.height, img.width

    for query in queries:
        if not query.strip():
            results[query] = []
            continue

        inputs = gdino_proc(images=img, text=query, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = gdino_model(**inputs)

        logits = outputs.logits.sigmoid()[0]
        boxes  = outputs.pred_boxes[0]
        scores = logits.max(dim=1).values if logits.dim() > 1 else logits

        mask = scores >= GDINO_TEXT_THRESHOLD
        filtered_boxes  = boxes[mask]
        filtered_scores = scores[mask]

        boxes_px = []
        for score, box_norm in zip(filtered_scores, filtered_boxes):
            # GroundingDINO returns (cx, cy, w, h) normalized
            cx_n, cy_n, bw_n, bh_n = box_norm.tolist()
            x1 = int((cx_n - bw_n / 2) * w)
            y1 = int((cy_n - bh_n / 2) * h)
            x2 = int((cx_n + bw_n / 2) * w)
            y2 = int((cy_n + bh_n / 2) * h)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            if score.item() >= GDINO_BOX_THRESHOLD:
                boxes_px.append({"box": [x1, y1, x2, y2], "score": score.item(), "label": query.strip()})

        results[query] = boxes_px
    return results


# ── Geometry helpers ──────────────────────────────────────
def centroid(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)

def horizontal_overlap(b1, b2):
    overlap = max(0, min(b1[2], b2[2]) - max(b1[0], b2[0]))
    max_w = max(b1[2]-b1[0], b2[2]-b2[0])
    return overlap / max_w if max_w > 0 else 0

def containment_ratio(inner, outer):
    xi1, yi1 = max(inner[0], outer[0]), max(inner[1], outer[1])
    xi2, yi2 = min(inner[2], outer[2]), min(inner[3], outer[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area = (inner[2]-inner[0]) * (inner[3]-inner[1])
    return inter / area if area > 0 else 0

def iou(b1, b2):
    xi1, yi1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    xi2, yi2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1+a2-inter) if (a1+a2-inter) > 0 else 0


# ── Spatial geometry verification ─────────────────────────
def verify_spatial(rel: str, subj_box: list, obj_box: list) -> dict:
    """Verify spatial relation from bboxes. Returns {verdict, evidence, correct_relation}."""
    rel_lower = rel.lower().strip()
    scx, scy = centroid(subj_box)
    ocx, ocy = centroid(obj_box)
    h_ov = horizontal_overlap(subj_box, obj_box)
    contain = containment_ratio(subj_box, obj_box)

    verdict, evidence, correct = None, "", None

    if rel_lower in ["left of", "to the left of"]:
        verdict = scx < ocx
        evidence = f"subj_cx={scx:.0f} {'<' if verdict else '>='} obj_cx={ocx:.0f}"
        if not verdict: correct = "to the right of"

    elif rel_lower in ["right of", "to the right of"]:
        verdict = scx > ocx
        evidence = f"subj_cx={scx:.0f} {'>' if verdict else '<='} obj_cx={ocx:.0f}"
        if not verdict: correct = "to the left of"

    elif rel_lower in ["above"]:
        verdict = scy < ocy
        evidence = f"subj_cy={scy:.0f} {'<' if verdict else '>='} obj_cy={ocy:.0f}"
        if not verdict: correct = "below"

    elif rel_lower in ["below", "beneath"]:
        verdict = scy > ocy
        evidence = f"subj_cy={scy:.0f} {'>' if verdict else '<='} obj_cy={ocy:.0f}"
        if not verdict: correct = "above"

    elif rel_lower in ["on", "on top of"]:
        verdict = scy < ocy and h_ov > 0.3
        evidence = f"subj_cy={scy:.0f} vs obj_cy={ocy:.0f}, h_overlap={h_ov:.2f}"
        if not verdict: correct = "under" if scy >= ocy else "not aligned"

    elif rel_lower in ["under", "underneath"]:
        verdict = scy > ocy and h_ov > 0.3
        evidence = f"subj_cy={scy:.0f} vs obj_cy={ocy:.0f}, h_overlap={h_ov:.2f}"
        if not verdict: correct = "on top of"

    elif rel_lower in ["inside", "in", "within", "inside of"]:
        verdict = contain > 0.6
        evidence = f"containment={contain*100:.0f}% (need >60%)"
        if not verdict: correct = "outside" if contain < 0.2 else "partially inside"

    elif rel_lower in ["outside", "outside of"]:
        verdict = contain < 0.2
        evidence = f"containment={contain*100:.0f}% (need <20% for outside)"
        if not verdict: correct = "inside"

    elif rel_lower in ["next to", "beside", "near", "close to"]:
        img_diag = math.sqrt(max(subj_box[2], obj_box[2])**2 + max(subj_box[3], obj_box[3])**2)
        dist = math.sqrt((scx-ocx)**2 + (scy-ocy)**2)
        rel_dist = dist / img_diag if img_diag > 0 else 1.0
        verdict = rel_dist < 0.3
        evidence = f"rel_dist={rel_dist:.2f} (threshold 0.3)"
        if not verdict: correct = "far from"

    elif rel_lower in ["far from", "away from"]:
        img_diag = math.sqrt(max(subj_box[2], obj_box[2])**2 + max(subj_box[3], obj_box[3])**2)
        dist = math.sqrt((scx-ocx)**2 + (scy-ocy)**2)
        rel_dist = dist / img_diag if img_diag > 0 else 1.0
        verdict = rel_dist >= 0.3
        evidence = f"rel_dist={rel_dist:.2f} (threshold 0.3)"
        if not verdict: correct = "near"

    else:
        evidence = f"No geometry rule for: {rel_lower}"

    return {"verdict": verdict, "evidence": evidence, "correct_relation": correct}


# ── Visualization ─────────────────────────────────────────
COLORS = ["#FF0000", "#00CC00", "#0066FF", "#FF9900", "#CC00FF", "#00CCCC", "#FF3399"]

def draw_detections(img, detections_dict):
    """Draw bounding boxes on image. Returns annotated PIL image."""
    draw_img = img.copy()
    draw = ImageDraw.Draw(draw_img)
    ci = 0
    for query, dets in detections_dict.items():
        color = COLORS[ci % len(COLORS)]
        ci += 1
        for d in dets:
            x1, y1, x2, y2 = d["box"]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            label = f"{query} ({d['score']:.2f})"
            draw.text((x1+2, y1-12), label, fill=color)
    return draw_img

print("✓ All functions defined")


In [ ]:
# ── Get test images ──────────────────────────────────────
import urllib.request, os
from PIL import Image, ImageDraw

TEST_DIR = "/content/test_images"
os.makedirs(TEST_DIR, exist_ok=True)

# Working COCO image URLs (using coco_synth_results from old run as reference)
TEST_IMAGES = {
    "cat_vehicle": {
        "queries": ["cat", "vehicle", "car"],
        "desc": "Cat and vehicle — test ON TOP OF vs INSIDE",
    },
    "person_objects": {
        "queries": ["person", "cell phone", "laptop", "bag"],
        "desc": "Person with objects — test spatial relations",
    },
    "animals_outdoor": {
        "queries": ["dog", "cat", "grass", "ground"],
        "desc": "Animals on ground — test ON, NEXT TO",
    },
    "street_scene": {
        "queries": ["motorcycle", "car", "truck", "road", "person"],
        "desc": "Street — test LEFT/RIGHT, FAR FROM",
    },
    "dining_scene": {
        "queries": ["cup", "bottle", "table", "plate", "food"],
        "desc": "Table with items — test ON, NEAR, INSIDE",
    },
}

# Alternative: Download from a more reliable source (huggingface datasets)
print("Downloading test images...")
try:
    from datasets import load_dataset
    coco_val = load_dataset("cocodataset", split="val", streaming=False, cache_dir="/tmp/coco")
    
    # Get first 5 images
    counter = 0
    for i, example in enumerate(coco_val):
        if counter >= 5:
            break
        try:
            img = example['image']
            name = list(TEST_IMAGES.keys())[counter]
            path = f"{TEST_DIR}/{name}.jpg"
            img.save(path)
            TEST_IMAGES[name]["path"] = path
            print(f"  ✓ {name}")
            counter += 1
        except:
            pass

except ImportError:
    print("⚠️  datasets library not available, using fallback approach")
    # Fallback: use simple URLs that work
    FALLBACK_URLS = {
        "cat_vehicle": "https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg",
        "person_objects": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg",
        "animals_outdoor": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/1280px-Dog_Breeds.jpg",
        "street_scene": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/City_road%2C_Dhaka%2C_Bangladesh_%28November_2014%29.jpg/1280px-City_road%2C_Dhaka%2C_Bangladesh_%28November_2014%29.jpg",
        "dining_scene": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg/1280px-Eq_it-na_pizza-margherita_sep2005_sml.jpg",
    }
    
    for name, url in list(FALLBACK_URLS.items())[:5]:
        path = f"{TEST_DIR}/{name}.jpg"
        if not os.path.exists(path):
            try:
                print(f"  Downloading {name}...", end=" ")
                urllib.request.urlretrieve(url, path, timeout=10)
                print("✓")
                TEST_IMAGES[name]["path"] = path
            except Exception as e:
                print(f"✗ ({e})")
        else:
            TEST_IMAGES[name]["path"] = path

# If downloads fail, create synthetic test images with objects
if not any("path" in v for v in TEST_IMAGES.values()):
    print("\n⚠️  Downloads failed, creating synthetic test images...")
    for name in TEST_IMAGES:
        # Create a simple synthetic image with colored rectangles
        img = Image.new("RGB", (640, 480), color="white")
        draw = ImageDraw.Draw(img)
        
        # Draw some colored boxes to simulate objects
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]
        for i, color in enumerate(colors):
            x1 = 50 + i * 100
            y1 = 100 + (i % 2) * 150
            x2 = x1 + 80
            y2 = y1 + 80
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1+10, y1+10), f"obj{i}", fill=color)
        
        path = f"{TEST_DIR}/{name}.jpg"
        img.save(path)
        TEST_IMAGES[name]["path"] = path
        print(f"  ✓ {name} (synthetic)")

print(f"\n✓ {sum(1 for v in TEST_IMAGES.values() if 'path' in v)} test images ready")
for name, info in TEST_IMAGES.items():
    if "path" in info:
        print(f"  {name}: {info['path']}")


In [ ]:
# ── Test 1: Object Detection ──────────────────────────────
# Verify GroundingDINO can find common objects

print("=" * 70)
print("TEST 1: Object Detection")
print("=" * 70)

all_detections = {}

for name, info in TEST_IMAGES.items():
    img = Image.open(info["path"]).convert("RGB")
    queries = ["cat", "person", "dog", "bus", "table", "chair", "couch",
               "laptop", "vehicle", "bottle", "cup", "food"]

    dets = run_gdino(img, queries)
    found = {q: d for q, d in dets.items() if d}
    all_detections[name] = {"img": img, "dets": dets, "found": found}

    print(f"\n📷 {name} ({info['desc'][:50]})")
    if found:
        for q, d_list in found.items():
            for d in d_list:
                x1,y1,x2,y2 = d["box"]
                w, h = x2-x1, y2-y1
                print(f"  ✓ {q:12s} box=[{x1:4d},{y1:4d},{x2:4d},{y2:4d}]  {w}x{h}px  score={d['score']:.3f}")
    else:
        print("  ✗ Nothing detected!")

# Show annotated images
fig, axes = plt.subplots(1, len(TEST_IMAGES), figsize=(20, 4))
for ax, (name, data) in zip(axes, all_detections.items()):
    annotated = draw_detections(data["img"], data["found"])
    ax.imshow(annotated)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("GroundingDINO Detections (colored boxes)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ── Test 2: Spatial Geometry Rules ────────────────────────
# For each image, pick detected object pairs and test ALL relations

print("=" * 70)
print("TEST 2: Spatial Geometry Verification")
print("=" * 70)

RELATIONS_TO_TEST = [
    "on", "on top of", "under", "underneath",
    "left of", "right of", "above", "below",
    "inside", "in", "outside",
    "next to", "near", "far from",
]

total_tests = 0
for name, data in all_detections.items():
    found = data["found"]
    labels = list(found.keys())
    if len(labels) < 2:
        print(f"\n📷 {name}: <2 objects detected, skipping")
        continue

    print(f"\n📷 {name} — detected: {', '.join(labels)}")

    # Test first pair of detected objects
    subj_label = labels[0]
    obj_label  = labels[1]
    subj_box = found[subj_label][0]["box"]
    obj_box  = found[obj_label][0]["box"]

    print(f"  Testing: {subj_label} [box={subj_box}] vs {obj_label} [box={obj_box}]")
    print(f"  {'Relation':<18s} {'Verdict':<12s} {'Evidence'}")
    print(f"  {'-'*18} {'-'*12} {'-'*40}")

    for rel in RELATIONS_TO_TEST:
        result = verify_spatial(rel, subj_box, obj_box)
        v = result["verdict"]
        tag = "✓ TRUE" if v == True else ("✗ FALSE" if v == False else "? NONE")
        print(f"  {rel:<18s} {tag:<12s} {result['evidence'][:55]}")
        total_tests += 1

print(f"\n{'='*70}")
print(f"Ran {total_tests} geometry checks across {len(all_detections)} images")
print(f"{'='*70}")


In [ ]:
# ── Test 3: Corruption Detection Simulation ───────────────
# Simulate the exact scenario from the synthetic test:
# Given a TRUE relation and a CORRUPTED relation, can geometry tell them apart?

print("=" * 70)
print("TEST 3: Can geometry detect corrupted relations?")
print("=" * 70)

CORRUPTION_TESTS = [
    # (subject_query, object_query, true_relation, corrupted_relation)
    ("cat",    "couch",   "on top of",  "under"),
    ("cat",    "couch",   "on top of",  "inside"),
    ("cat",    "laptop",  "on",         "under"),
    ("cat",    "laptop",  "near",       "far from"),
    ("person", "bus",     "near",       "far from"),
    ("person", "bus",     "left of",    "right of"),
    ("person", "chair",   "on",         "under"),
    ("dog",    "couch",   "on",         "under"),
    ("dog",    "couch",   "on top of",  "inside"),
    ("cup",    "table",   "on",         "under"),
    ("bottle", "table",   "on",         "inside"),
]

results_table = []

for subj_q, obj_q, true_rel, corrupt_rel in CORRUPTION_TESTS:
    # Find an image where both objects are detected
    found_img = None
    for name, data in all_detections.items():
        f = data["found"]
        if subj_q in f and obj_q in f:
            found_img = (name, f[subj_q][0]["box"], f[obj_q][0]["box"])
            break

    if not found_img:
        results_table.append({
            "test": f"({subj_q}, {true_rel}, {obj_q}) → {corrupt_rel}",
            "image": "—",
            "true_verdict": "—",
            "corrupt_verdict": "—",
            "detected": "NO OBJECTS",
        })
        continue

    img_name, subj_box, obj_box = found_img

    true_result    = verify_spatial(true_rel,    subj_box, obj_box)
    corrupt_result = verify_spatial(corrupt_rel, subj_box, obj_box)

    true_v = true_result["verdict"]
    corrupt_v = corrupt_result["verdict"]

    # Detection success = true relation is True AND corrupted is False
    if true_v == True and corrupt_v == False:
        detected = "✅ YES"
    elif true_v == True and corrupt_v == True:
        detected = "⚠️  BOTH TRUE"
    elif true_v == False:
        detected = "❌ TRUE FAILED"
    elif corrupt_v is None or true_v is None:
        detected = "❓ NO RULE"
    else:
        detected = "❌ MISSED"

    results_table.append({
        "test": f"({subj_q}, {true_rel}, {obj_q}) → {corrupt_rel}",
        "image": img_name,
        "true_verdict": f"{true_v} — {true_result['evidence'][:40]}",
        "corrupt_verdict": f"{corrupt_v} — {corrupt_result['evidence'][:40]}",
        "detected": detected,
    })

# Print results table
print(f"\n{'Test':<50s} {'Image':<16s} {'Detected?'}")
print("-" * 80)
for r in results_table:
    print(f"{r['test']:<50s} {r['image']:<16s} {r['detected']}")

print(f"\n{'='*70}")
detected_count = sum(1 for r in results_table if "YES" in r["detected"])
total = len(results_table)
no_obj = sum(1 for r in results_table if "NO OBJECTS" in r["detected"])
print(f"Detected: {detected_count}/{total-no_obj} (excluding {no_obj} with no detections)")
print(f"{'='*70}")

# Also print detailed verdicts for debugging
print("\nDetailed verdicts:")
for r in results_table:
    print(f"\n  {r['test']}")
    print(f"    True:    {r['true_verdict']}")
    print(f"    Corrupt: {r['corrupt_verdict']}")
